# 06 — Data Quality & BI Reporting

**Goal:** Validate the star schema created in 05, produce a city-level beverage sales mart, and prepare two CSV datasets for Power BI.

**Inputs:** Existing `workspace.gold` tables from 05. No CSV reloads or Silver reprocessing.

**Outputs:**
- `workspace.gold.data_quality_results` (quality check results)
- `workspace.gold.mart_monthly_city_beverage_kpis` (city × month)
- Optional CSV exports in `/Volumes/workspace/gold/bi_exports/`

This is a reproducible **full-refresh reporting step**, not an automated production monitoring system or a completed Power BI dashboard.

## 1. Read the existing Gold tables

Use the store-month mart, two fact tables, and dimensions built in 05. All further work is based on these trusted outputs.

In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.gold"

store_month = spark.table(f"{GOLD}.mart_monthly_beverage_kpis")
fact_sales = spark.table(f"{GOLD}.fact_sales")
fact_transactions = spark.table(f"{GOLD}.fact_transactions")
dim_date = spark.table(f"{GOLD}.dim_date")
dim_store = spark.table(f"{GOLD}.dim_store")
dim_product_family = spark.table(f"{GOLD}.dim_product_family")

print("Existing Gold tables loaded.")
print("Store-month rows:", store_month.count())

## 2. Build a city-month reporting mart

**Grain:** one row per `month_start × state × city`. The existing store-month table stays unchanged.

The city-level average uses **weighted store-days** (`total beverage quantity / recorded store-days`). Never sum or simply average store-level averages. The same rule applies to recorded transactions: missing transaction days are **not** treated as zero.

In [0]:
city_month = (
    store_month
    .groupBy("month_start", "year", "month", "state", "city")
    .agg(
        F.countDistinct("store_nbr").alias("store_count"),
        F.sum("total_beverage_sales_qty").alias("total_beverage_sales_qty"),
        F.sum("sales_record_days").alias("sales_store_days"),
        F.sum("promotion_days").alias("promotion_store_days"),
        F.sum("holiday_sales_days").alias("holiday_store_days"),
        F.sum("total_recorded_transactions").alias("total_recorded_transactions"),
        F.sum("transaction_record_days").alias("transaction_store_days")
    )
    .withColumn(
        "avg_beverage_qty_per_store_day",
        F.round(F.col("total_beverage_sales_qty") / F.col("sales_store_days"), 2)
    )
    .withColumn(
        "avg_transactions_per_recorded_store_day",
        F.when(
            F.col("transaction_store_days") > 0,
            F.round(
                F.col("total_recorded_transactions") / F.col("transaction_store_days"), 2
            )
        )
    )
)

city_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{GOLD}.mart_monthly_city_beverage_kpis"
)
print("Saved:", f"{GOLD}.mart_monthly_city_beverage_kpis")
display(city_month.orderBy("month_start", "state", "city").limit(10))

## 3. Data quality audit

This audit checks important reporting assumptions: unique business keys, non-null critical fields, dimension references, nonnegative quantities, and reconciliation between facts, store-month metrics, and city-month metrics.

Checks report **issues found**, not an invented data quality score. The only table this audit overwrites is its own result table.

In [0]:
checks = []

def check(name, issues, description):
    issue_count = int(issues)
    status = "PASS" if issue_count == 0 else "FAIL"
    checks.append((name, status, issue_count, description))
    print(f"{status}: {name} ({issue_count} issues)")

def duplicate_keys(df, keys):
    return df.groupBy(*keys).count().filter(F.col("count") > 1).count()

check("Unique sales business keys", duplicate_keys(fact_sales, ["date", "store_nbr", "family"]),
      "One sales record per date, store, product family")
check("Unique store-month keys", duplicate_keys(store_month, ["month_start", "store_nbr"]),
      "One row per store per month")
check("Unique city-month keys", duplicate_keys(city_month, ["month_start", "state", "city"]),
      "One row per state, city and month")

bad_sales = fact_sales.filter(
    F.col("date").isNull() |
    F.col("store_nbr").isNull() |
    F.col("family").isNull() |
    F.col("sales_qty").isNull() |
    (F.col("sales_qty") < 0) |
    F.col("promoted_item_count").isNull() |
    (F.col("promoted_item_count") < 0)
).count()
check("Valid sales values", bad_sales, "No missing keys, missing quantities or negative values")

for label, keys, fact, dim in [
    ("Sales store references", ["store_nbr"], fact_sales, dim_store),
    ("Sales date references", ["date"], fact_sales, dim_date),
    ("Sales family references", ["family"], fact_sales, dim_product_family),
    ("Transaction store references", ["store_nbr"], fact_transactions, dim_store),
    ("Transaction date references", ["date"], fact_transactions, dim_date),
]:
    orphan_keys = (
        fact.select(*keys).distinct()
        .join(dim.select(*keys).distinct(), keys, "left_anti")
        .count()
    )
    check(label, orphan_keys, "All keys reference existing dimension members")

invalid_days = store_month.filter(
    (F.col("sales_record_days") <= 0) |
    (F.col("total_beverage_sales_qty") < 0) |
    (F.col("transaction_record_days") < 0) |
    (F.col("total_recorded_transactions") < 0)
).count()
check("Valid reporting measures", invalid_days,
      "Positive observed sales days and nonnegative recorded measures")

## 4. Reconcile data totals and save the audit

Reconciliation catches accidental multiplication of records during joins. Since the mart uses beverage quantities only, its totals should equal the matching beverage rows in `fact_sales`.

A single record with NULL transaction counts is not automatically a data quality failure: the source transactions table has incomplete store-date coverage.

In [0]:
beverage_qty = (
    fact_sales.filter(F.col("family") == "BEVERAGES")
    .agg(F.sum("sales_qty").alias("qty")).first()["qty"]
)
store_month_qty = store_month.agg(F.sum("total_beverage_sales_qty").alias("qty")).first()["qty"]
city_month_qty = city_month.agg(F.sum("total_beverage_sales_qty").alias("qty")).first()["qty"]

check("Sales-to-store-month reconciliation", int(abs(beverage_qty - store_month_qty) >= 0.01),
      "Beverage quantities agree with fact_sales")
check("Store-month-to-city-month reconciliation", int(abs(store_month_qty - city_month_qty) >= 0.01),
      "Rollup totals agree between store and city marts")

quality_results = (
    spark.createDataFrame(checks, ["check_name", "status", "issue_count", "description"])
    .withColumn("checked_at", F.current_timestamp())
)
quality_results.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{GOLD}.data_quality_results"
)

display(quality_results.orderBy("status", "check_name"))
failed = [row[0] for row in checks if row[1] == "FAIL"]
if failed:
    raise ValueError("Data quality checks failed: " + ", ".join(failed))
print("All data quality checks passed.")

## 5. Export two small datasets for Power BI (optional)

The **store-month** export is the primary dashboard dataset. The **city-month** export is an independent, already aggregated view. **Do not append them together** (that would double-count sales).

The code tries to create a dedicated Unity Catalog Volume for CSV files. If your workspace lacks permissions, the Gold Delta tables will still remain available for export or direct Power BI connection.

In [0]:
# These aggregated tables are small enough for pandas CSV export.
# The export is optional; it should not stop the completed data pipeline.
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {GOLD}.bi_exports")
    export_dir = "/Volumes/workspace/gold/bi_exports"

    store_month.orderBy("month_start", "store_nbr").toPandas().to_csv(
        f"{export_dir}/store_monthly_beverages.csv", index=False
    )
    city_month.orderBy("month_start", "state", "city").toPandas().to_csv(
        f"{export_dir}/city_monthly_beverages.csv", index=False
    )
    print("CSV exports saved to:", export_dir)
except Exception as exc:
    print("Optional CSV export skipped:", type(exc).__name__, str(exc)[:300])
    print("Gold tables are still available; you can export them manually.")

## 6. Power BI report plan

Import `store_monthly_beverages.csv` (or directly connect to `workspace.gold.mart_monthly_beverage_kpis` when your workspace allows it). In Power BI, make sure `month_start` is a **Date** and numeric columns are numeric.

Suggested report:

| Visual | Fields |
|---|---|
| KPI cards | Total beverage quantity; recorded transactions; active stores |
| Line chart | `month_start` vs total beverage quantity |
| Bar chart | `city` vs total beverage quantity |
| Slicers | year / month_start / state / city / store_nbr |

**DAX measures** (assuming the table is named `store_monthly_beverages`):

```dax
Total Beverage Qty = SUM('store_monthly_beverages'[total_beverage_sales_qty])

Avg Beverage Qty per Recorded Store-Day =
DIVIDE(
    SUM('store_monthly_beverages'[total_beverage_sales_qty]),
    SUM('store_monthly_beverages'[sales_record_days])
)

Total Recorded Transactions = SUM('store_monthly_beverages'[total_recorded_transactions])

Active Stores = DISTINCTCOUNT('store_monthly_beverages'[store_nbr])
```

**Interpretation:** Beverage `sales` is a quantity, not revenue. Transaction records have missing store-days; `total_recorded_transactions` means observed transactions only. Do not sum `avg_daily_beverage_sales_qty` or `avg_transactions_per_recorded_day` across stores. There is no true revenue, profit, or inventory value in this project.

**After finishing in Power BI Desktop:** Save a `.pbix` locally and add a screenshot to your GitHub `dashboards/` folder. This Notebook prepares the data but **does not create the Power BI dashboard automatically**.

## 7. Deliverables

- `workspace.gold.data_quality_results`: the latest validation results.
- `workspace.gold.mart_monthly_city_beverage_kpis`: aggregate view by state, city and month.
- `workspace.gold.mart_monthly_beverage_kpis`: the existing store-month reporting table from 05.
- Optional CSV files under `workspace.gold.bi_exports` (a **volume**, not a Delta table).

**Scope:** No machine learning model, incremental data ingestion, scheduled refreshes, or Power BI `.pbix` file is implemented here.